In [ ]:
import sys
sys.path.append('..')
import numpy as np

from util.plot import start_end_subplot

from model import models, networks
from model.models import EDMPrecond,FMCond
from model.networks import MLP,MLP_dd
import trimesh

from util.mesh_utils import *

device='cuda:0'
torch.cuda.set_device(device)

PLOT=True

In [ ]:
lm=np.array( [412, 5891,6593,3323,2119] )
NUM_POINTS_INFERENCE=10000
NUM_STEP=64

SOURCE="tr_reg_080"
TARGET="tr_reg_093"

In [ ]:
mesh1= normalize_mesh(trimesh.load('../../GeomDist/faust/test_set/'+SOURCE+'.off'))
mesh2= normalize_mesh(trimesh.load('../../GeomDist/faust/test_set/'+TARGET+'.off'))
mesh1.vertices=mesh1.vertices#@np.array([[1,0,0],[0,1,0],[0,0,-1]])
model = FMCond(channels=3+len(lm) , network=MLP_dd(channels=3+len(lm)).to(device))   
model.to(device)
model.load_state_dict(torch.load('../output/'+SOURCE+'_ldmk/checkpoint-9999.pth', map_location=device,weights_only=False)['model'], strict=True)


model2 = FMCond(channels=3+len(lm) , network=MLP_dd(channels=3+len(lm)).to(device))   
model2.to(device)
model2.load_state_dict(torch.load('../output/'+TARGET+'_ldmk/checkpoint-9999.pth', map_location=device,weights_only=False)['model'], strict=True)

model.eval()
model2.eval()

v1=torch.tensor(mesh1.vertices).float().to(device)
v2=torch.tensor(mesh2.vertices).float().to(device)



In [ ]:
class Args:
    def __init__(self, num_points_train, embedding):
        self.num_points_train = num_points_train
        self.embedding = embedding

args = Args(num_points_train=10000, embedding="landmark_feat")
args.landmarks = lm
emb1=generate_embeddings(mesh1,args)
enmb2=generate_embeddings(mesh2,args)

In [ ]:
emb1_pullback=model.inverse(emb1)
emb2_pullback=model2.inverse(enmb2)

In [ ]:
noise = model.inverse(samples=emb1, num_steps=NUM_STEP)
sample = model2.sample(noise=noise, num_steps=NUM_STEP)



start_end_subplot(emb1[:,:3].cpu(),sample[:,:3].cpu() ,show=True)

